# Part A — Two-Player Crossword Puzzle using Minimax and Decision-Tree Visualization
This notebook implements the Part A problem statement — a crossword-style two-player game where the computer uses the Minimax algorithm with alpha–beta pruning to make decisions. It also visualizes the decision tree using `matplotlib` and `networkx`. The human interacts via console inputs.

In [ ]:
import math
import copy
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import networkx as nx

@dataclass
class Slot:
    id:int
    length:int
    correct_word:str
    filled:bool=False

@dataclass
class GameState:
    slots:Dict[int,Slot]
    words:List[str]
    human_score:int
    comp_score:int
    current_player:str
    forced_word:Optional[str]=None

def initial_game_state():
    slots = {
        1:Slot(1,4,'crow'),
        2:Slot(2,3,'bat'),
        3:Slot(3,3,'dog'),
        4:Slot(4,4,'wolf'),
        5:Slot(5,8,'elephant')
    }
    return GameState(slots, list(s.correct_word for s in slots.values()), 0, 0, 'H')

def available_slots(state):
    return [s for s in state.slots.values() if not s.filled]

def apply_move(state, player, word, slot_id):
    new = copy.deepcopy(state)
    slot = new.slots[slot_id]
    if word == slot.correct_word:
        if player=='H': new.human_score += len(word)
        else: new.comp_score += len(word)
        slot.filled = True
        new.words.remove(word)
        new.forced_word=None
        new.current_player = ('C' if player=='H' else 'H')
    else:
        if player=='H': new.human_score -= 1; new.current_player='C'
        else: new.comp_score -= 1; new.current_player='H'
        new.forced_word = word
    return new

def is_terminal(state):
    return all(s.filled for s in state.slots.values())

def evaluate(state):
    diff = state.comp_score - state.human_score
    remaining = sum(len(w) for w in state.words)
    sign = 1 if state.current_player=='C' else -1
    return diff + 0.5*remaining*sign

def legal_actions(state):
    slots = available_slots(state)
    if state.forced_word:
        return [(state.forced_word, s.id) for s in slots]
    acts=[]
    for w in state.words:
        for s in slots:
            acts.append((w,s.id))
    return acts

def minimax(state, depth, alpha, beta, graph, parent=None, label='root'):
    if is_terminal(state) or depth==0:
        val = evaluate(state)
        graph.add_node(label, value=round(val,2))
        if parent: graph.add_edge(parent,label)
        return val, None

    player = state.current_player
    graph.add_node(label, value=None)
    if parent: graph.add_edge(parent,label)
    best_action=None

    if player=='C':
        value=-math.inf
        for i,action in enumerate(legal_actions(state)):
            child=apply_move(state,'C',*action)
            child_label=f"{label}-C{i}:{action}"
            val,_=minimax(child,depth-1,alpha,beta,graph,label,child_label)
            if val>value:
                value=val; best_action=action
            alpha=max(alpha,value)
            if alpha>=beta: break
        graph.nodes[label]['value']=round(value,2)
        return value,best_action
    else:
        value=math.inf
        for i,action in enumerate(legal_actions(state)):
            child=apply_move(state,'H',*action)
            child_label=f"{label}-H{i}:{action}"
            val,_=minimax(child,depth-1,alpha,beta,graph,label,child_label)
            if val<value:
                value=val; best_action=action
            beta=min(beta,value)
            if alpha>=beta: break
        graph.nodes[label]['value']=round(value,2)
        return value,best_action

def plot_tree(graph):
    pos = nx.spring_layout(graph, seed=42)
    labels = {n:f"{n.split(':')[-1]}\n{graph.nodes[n].get('value','')}" for n in graph.nodes}
    plt.figure(figsize=(12,6))
    nx.draw(graph, pos, with_labels=False, node_color='lightblue', node_size=1500)
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)
    plt.title('Minimax Decision Tree')
    plt.show()

def computer_move(state, depth=3):
    graph = nx.DiGraph()
    _, action = minimax(state, depth, -math.inf, math.inf, graph)
    plot_tree(graph)
    return action

def print_board(state):
    print('\nBoard:')
    for sid,s in state.slots.items():
        txt = s.correct_word if s.filled else '___'
        print(f"Slot {sid}({s.length}): {txt}")
    print(f"Words: {state.words}")
    print(f"Scores H:{state.human_score} C:{state.comp_score}")

def human_input(state):
    while True:
        try:
            w = state.forced_word if state.forced_word else input('Word:').strip()
            sid = int(input('Slot id:'))
            if w not in state.words or sid not in state.slots or state.slots[sid].filled:
                print('Invalid, try again.')
                continue
            return w,sid
        except: print('Bad input.')

def play():
    s = initial_game_state()
    print('Two-Player Crossword using Minimax')
    print_board(s)
    while not is_terminal(s):
        if s.current_player=='H':
            print('\nHuman turn')
            w,sid=human_input(s)
            s=apply_move(s,'H',w,sid)
            print_board(s)
        else:
            print('\nComputer turn (Minimax searching...)')
            w,sid=computer_move(s,depth=3)
            print(f"Computer chose ({w},{sid})")
            s=apply_move(s,'C',w,sid)
            print_board(s)
    print('\nGame Over! Final Scores: H',s.human_score,' C',s.comp_score)
    print('Winner:', 'HUMAN' if s.human_score>s.comp_score else 'COMPUTER' if s.comp_score>s.human_score else 'TIE')


In [ ]:
play()  # Run this to play interactively